# Simplicits with MPM (Material Point Method)

This notebook demonstrates one-way coupling between Simplicits soft-body simulation
and MPM granular materials (sand). Two deformable cubes act as static obstacles inside
the MPM model — each frame, Simplicits particles are mirrored into the MPM state via
`move_static_particles_kernel` so the sand sees the current cube positions.

## Imports

We import Kaolin, Newton, Warp, and **k3d** for interactive 3D visualization.
`SolverImplicitMPM` drives the sand simulation; `SimplicitsModelBuilder` /
`SimplicitsSolver` drive the soft-body cubes. `ParticleFlags` marks particles
as active or static.

In [ ]:
import os
import threading

import k3d
import numpy as np
import torch
import warp as wp
from IPython.display import display
from ipywidgets import Button, VBox

import newton
from newton import ModelBuilder, Contacts
from newton.solvers import SolverImplicitMPM
from newton._src.geometry import ParticleFlags

import kaolin
import kaolin.physics.simplicits
from examples.tutorial.tutorial_common import COMMON_DATA_DIR
from kaolin.experimental.newton.builder import SimplicitsModelBuilder
from kaolin.experimental.newton.solver import SimplicitsSolver

## Configuration

Key constants:
- **`SOFT_YOUNGS_MODULUS` / `POISSON_RATIO` / `DENSITY`** — elastic material parameters.
- **`FRAME_DT`** — simulation timestep. MPM runs `sim_substeps=10` sub-steps per frame.
- **`COLLISION_PARTICLE_RADIUS`** — radius used for Simplicits self-collision detection.
- **`MPM_VOXEL_SIZE`** — grid cell size for the MPM solver.
- **`SAND_BOUNDS_LO/HI`** — bounding box for sand particle spawn region.
- **`CUBE_TRANSFORM_1/2`** — initial 4×4 transforms for the two soft cubes.

In [ ]:
# Simulation parameters
FRAME_DT = 1.0 / 100.0
FLOOR_PLANE = 0.0

# Simplicits parameters
SOFT_YOUNGS_MODULUS = 1e4
POISSON_RATIO = 0.45
DENSITY = 500
APPROX_VOLUME = 0.5
NUM_SAMPLES = 10000

# Contact parameters
SELF_CONTACT_RADIUS = 0.01
SELF_CONTACT_MARGIN = 0.01
SOFT_CONTACT_KE = 100
SOFT_CONTACT_KD = 2e-3
SELF_CONTACT_FRICTION = 0.25
SOFT_CONTACT_MAX = 1000000

# Collision parameters
COLLISION_PARTICLE_RADIUS = 0.05
DETECTION_RATIO = 1.5
IMPENETRABLE_BARRIER_RATIO = 0.25
COLLISION_PENALTY = 1000.0
MAX_CONTACT_PAIRS = 10000
COLLISION_FRICTION = 0.5

# MPM parameters
MPM_VOXEL_SIZE = 0.07
MPM_TOLERANCE = 1.0e-6
MPM_MAX_ITERATIONS = 250
MPM_PARTICLE_KE = 1.0e15
MPM_PARTICLE_KD = 0.01
MPM_PARTICLE_MU = 0.5
PARTICLES_PER_CELL = 3

# Inter-model collision parameters
MAX_INTER_CONTACTS = 20000
INTER_COLLISION_STIFFNESS = 1e4
INTER_COLLISION_DAMPING = 100.0
CONTACT_FORCE_SCALE = 1.0
MAX_VELOCITY_CHANGE = 10.0

# Sand spawn parameters
SAND_BOUNDS_LO = np.array([0.25, -0.5, 1.5])
SAND_BOUNDS_HI = np.array([0.75, 0.5, 1.75])
SAND_DENSITY = 2500.0

# Simplicits spawn transforms
CUBE_TRANSFORM_1 = torch.tensor([
    [1, 0, 0, 0.4],
    [0, 1, 0, 0.1],
    [0, 0, 1, 0.3],
    [0, 0, 0, 1]
], dtype=torch.float32, device='cuda')

CUBE_TRANSFORM_2 = torch.tensor([
    [1, 0, 0, 0.4],
    [0, 1, 0, 0.1],
    [0, 0, 1, 1.0],
    [0, 0, 0, 1]
], dtype=torch.float32, device='cuda')

## Warp Kernels
**`move_static_particles_kernel`** — each frame, copies Simplicits particle positions
   and velocities into the MPM static obstacle slots.

In [ ]:
wp.clear_kernel_cache()

@wp.kernel
def move_static_particles_kernel(
    static_idx: wp.array(dtype=int),
    particle_q: wp.array(dtype=wp.vec3),
    particle_qd: wp.array(dtype=wp.vec3),
    sim_pts: wp.array(dtype=wp.vec3),
    sim_vel: wp.array(dtype=wp.vec3),
):
    """Update static MPM particles to mirror current Simplicits particle positions."""
    tid = wp.tid()
    st_idx = static_idx[tid]
    particle_q[st_idx] = sim_pts[tid]
    particle_qd[st_idx] = sim_vel[tid]

## Simplicits Object and Model

`get_simplicits_cube()` loads a unit cube mesh, scales it by 0.5, samples interior
points, and creates a affinely deformable `SimplicitsObject`. It returns both `sim_obj` (for physics)
and `mesh` (for k3d visualization).

Two cubes are added at `CUBE_TRANSFORM_1` and `CUBE_TRANSFORM_2`. Their object IDs
(`obj_id_0`, `obj_id_1`) are captured for later use in `get_object_deformed_pts`.

In [ ]:
def get_simplicits_cube():
    """Load cube mesh and create a single handle Simplicits object. Returns (sim_obj, mesh)."""
    mesh_path = os.path.join(COMMON_DATA_DIR, "meshes")
    mesh = kaolin.io.import_mesh(
        mesh_path + "/cube.obj", triangulate=True
    ).to('cuda')
    mesh.vertices = kaolin.ops.pointcloud.center_points(
        mesh.vertices.unsqueeze(0), normalize=True
    ).squeeze(0) * 0.5
    orig_vertices = mesh.vertices.clone()

    uniform_pts = torch.rand(NUM_SAMPLES, 3, device='cuda') * (
        orig_vertices.max(dim=0).values - orig_vertices.min(dim=0).values
    ) + orig_vertices.min(dim=0).values

    yms = torch.full((NUM_SAMPLES,), SOFT_YOUNGS_MODULUS, device='cuda')
    prs = torch.full((NUM_SAMPLES,), POISSON_RATIO, device='cuda')
    rhos = torch.full((NUM_SAMPLES,), DENSITY, device='cuda')

    sim_obj = kaolin.physics.simplicits.SimplicitsObject.create_rigid(
        uniform_pts, yms, prs, rhos, APPROX_VOLUME
    ) # create_rigid constructs an *deformable* object with a single handle for fast construction. At higher YM values it acts as rigid.
    return sim_obj, mesh


# Create Simplicits objects
sim_obj, mesh = get_simplicits_cube()
orig_vertices = mesh.vertices.clone()

# Build Simplicits model
simplicits_builder = SimplicitsModelBuilder()
simplicits_builder.add_ground_plane()

simplicits_builder.add_simplicits_object(sim_obj, num_qp=2000, init_transform=CUBE_TRANSFORM_1, renderable_pts=orig_vertices.clone().detach())
simplicits_builder.add_simplicits_object(sim_obj, num_qp=2000, init_transform=CUBE_TRANSFORM_2, renderable_pts=orig_vertices.clone().detach())

simplicits_builder.add_simplicits_collisions(
    collision_particle_radius=COLLISION_PARTICLE_RADIUS,
    detection_ratio=DETECTION_RATIO,
    impenetrable_barrier_ratio=IMPENETRABLE_BARRIER_RATIO,
    collision_penalty=COLLISION_PENALTY,
    max_contact_pairs=MAX_CONTACT_PAIRS,
    friction=COLLISION_FRICTION
)

simplicits_model = simplicits_builder.finalize()
simplicits_state_0 = simplicits_model.state()
simplicits_state_1 = simplicits_model.state()

simplicits_solver = SimplicitsSolver(simplicits_model)
simplicits_model.simplicits_scene.max_newton_steps = 5
simplicits_model.simplicits_scene.max_ls_steps = 10
simplicits_model.simplicits_scene.newton_hessian_regularizer = 1e-4

print(f"Simplicits model: {simplicits_model.particle_count} particles")

## MPM Setup

The MPM model holds two types of particles:

1. **Sand particles** (indices `0 : mpm_static_start_idx`) — active granular material
   spawned in `SAND_BOUNDS_LO/HI` with `PARTICLES_PER_CELL` per voxel cell.
2. **Static Simplicits obstacle particles** (indices `mpm_static_start_idx :`) — copies
   of Simplicits integration points added with `flags=0` (inactive for MPM dynamics).
   Each frame, `move_static_particles_kernel` updates their positions to match the
   current Simplicits state so the MPM solver sees updated obstacles.

In [ ]:
def _spawn_particles(builder, voxel_size, bounds_lo, bounds_hi, density, flags):
    """Spawn particles on a regular grid within a bounding box."""
    res = np.array(
        np.ceil(PARTICLES_PER_CELL * (bounds_hi - bounds_lo) / voxel_size), dtype=int
    )
    px = np.linspace(bounds_lo[0], bounds_hi[0], res[0] + 1)
    py = np.linspace(bounds_lo[1], bounds_hi[1], res[1] + 1)
    pz = np.linspace(bounds_lo[2], bounds_hi[2], res[2] + 1)
    points = np.stack(np.meshgrid(px, py, pz)).reshape(3, -1).T
    cell_size = (bounds_hi - bounds_lo) / res
    cell_volume = np.prod(cell_size)
    radius = np.max(cell_size) * 1.0
    print(f"mpm particleradius: {radius}")
    mass = np.prod(cell_volume) * density
    rng = np.random.default_rng(42)
    points += 2.0 * radius * (rng.random(points.shape) - 0.5)
    first_id = len(builder.particle_q)
    builder.add_particles(
        pos=[p for p in points],
        vel=[(0.0, 0.0, 0.0)] * points.shape[0],
        mass=[mass] * points.shape[0],
        radius=[radius] * points.shape[0],
        flags=[flags] * points.shape[0],
    )
    return np.arange(first_id, first_id + points.shape[0], dtype=int)


def emit_particles(builder, voxel_size):
    """Spawn sand particles in SAND_BOUNDS region."""
    return _spawn_particles(
        builder, voxel_size, SAND_BOUNDS_LO, SAND_BOUNDS_HI, SAND_DENSITY, ParticleFlags.ACTIVE
    )


sim_substeps = 10
frame_dt = FRAME_DT
sim_dt = frame_dt / sim_substeps

mpm_builder = ModelBuilder()
SolverImplicitMPM.register_custom_attributes(mpm_builder)
mpm_builder.add_ground_plane()

sand_particles_ids = emit_particles(mpm_builder, voxel_size=MPM_VOXEL_SIZE)

# Mirror Simplicits particles as static obstacles in MPM
sim_pts = simplicits_model.sim_z_to_full(simplicits_state_0.sim_z).numpy()
mpm_static_start_idx = len(mpm_builder.particle_q)

mpm_builder.add_particles(
    pos=[p for p in sim_pts],
    vel=[(0.0, 0.0, 0.0)] * sim_pts.shape[0],
    mass=[1.0] * sim_pts.shape[0],
    radius=[0.1] * sim_pts.shape[0],
    flags=[0] * sim_pts.shape[0],
)

mpm_static_idx_end = len(mpm_builder.particle_q)
mpm_static_idx = wp.array(
    np.arange(mpm_static_start_idx, mpm_static_idx_end, dtype=int),
    dtype=wp.int32, device='cuda'
)

mpm_model = mpm_builder.finalize()

mpm_options = SolverImplicitMPM.Config()
mpm_options.voxel_size = MPM_VOXEL_SIZE
mpm_options.tolerance = MPM_TOLERANCE
mpm_options.max_iterations = MPM_MAX_ITERATIONS

mpm_state_0 = mpm_model.state()
mpm_state_1 = mpm_model.state()

mpm_solver = SolverImplicitMPM(mpm_model, mpm_options)

mpm_model.particle_ke = MPM_PARTICLE_KE
mpm_model.particle_kd = MPM_PARTICLE_KD
mpm_model.particle_mu = MPM_PARTICLE_MU

print(f"MPM model: {mpm_model.particle_count} particles "
      f"({mpm_static_start_idx} sand + {mpm_static_idx_end - mpm_static_start_idx} static)")

## Simulation Loop

Each call to `simulate()` advances the scene by one frame:

1. **Simplicits step** — one Newton solve advances the two soft cubes by `frame_dt`.
2. **Static particle update** — `move_static_particles_kernel` copies the new Simplicits
   particle positions/velocities into the MPM static obstacle slots.
3. **MPM substeps** — `sim_substeps=10` implicit MPM steps advance the sand by `sim_dt` each.

In [ ]:
def simulate():
    global simplicits_state_0, simplicits_state_1, mpm_state_0, mpm_state_1

    # Simplicits step (one big step per frame)
    simplicits_state_0.clear_forces()
    simplicits_state_1.clear_forces()
    contacts = simplicits_model.collide(simplicits_state_0)
    simplicits_solver.step(
        simplicits_state_0, simplicits_state_1,
        None, contacts, frame_dt
    )
    simplicits_state_0, simplicits_state_1 = simplicits_state_1, simplicits_state_0

    # Update MPM static obstacle particles to match current Simplicits positions
    wp_sim_pts = simplicits_model.sim_z_to_full(simplicits_state_1.sim_z)
    wp_sim_vel = simplicits_model.sim_z_dot_to_full(simplicits_state_0.sim_z_dot)
    wp.launch(
        kernel=move_static_particles_kernel,
        dim=mpm_static_idx.shape[0],
        inputs=[
            mpm_static_idx,
            mpm_state_0.particle_q,
            mpm_state_0.particle_qd,
            wp_sim_pts,
            wp_sim_vel,
        ],
    )

    # MPM substeps
    for _ in range(sim_substeps):
        mpm_state_0.clear_forces()
        mpm_state_1.clear_forces()
        mpm_solver.step(mpm_state_0, mpm_state_1, None, None, sim_dt)
        mpm_state_0, mpm_state_1 = mpm_state_1, mpm_state_0

## Visualization with k3d

Two k3d objects are rendered:

- **`mesh_soft`** (blue) — the two Simplicits cubes combined into a single mesh.
  Updated via `get_object_deformed_pts` each frame.
- **`points_sand`** (yellow) — the active sand particles
  (`mpm_state_0.particle_q[:mpm_static_start_idx]`).
  The static Simplicits obstacle copies are hidden.

Play/Stop/Reset buttons control the background simulation thread.

In [ ]:
orig_vertices_np = orig_vertices.cpu().numpy().astype(np.float32)
orig_faces_np = mesh.faces.cpu().numpy().astype(np.uint32)
num_verts = orig_vertices_np.shape[0]

combined_faces_np = np.concatenate(
    [orig_faces_np, orig_faces_np + num_verts]
).astype(np.uint32)

plot = k3d.plot(camera_auto_fit=False)
plot.camera = [3, 3, 3,  0, 0, 0,  0, 0, 1]

mesh_soft = k3d.mesh(
    np.concatenate([orig_vertices_np, orig_vertices_np]).astype(np.float32),
    combined_faces_np,
    color=0x3399ff
)

sand_pts_np = mpm_state_0.particle_q.numpy()[:mpm_static_start_idx].astype(np.float32)
points_sand = k3d.points(sand_pts_np, color=0xffcc00, point_size=0.05)

plot += mesh_soft
plot += points_sand

# Floor plane at FLOOR_PLANE height
_s = 3.0  # half-extent
floor_verts = np.array([
    [-_s, -_s, FLOOR_PLANE],
    [ _s, -_s, FLOOR_PLANE],
    [ _s,  _s, FLOOR_PLANE],
    [-_s,  _s, FLOOR_PLANE],
], dtype=np.float32)
floor_faces = np.array([[0, 1, 2], [0, 2, 3]], dtype=np.uint32)
mesh_floor = k3d.mesh(floor_verts, floor_faces, color=0xcccccc, opacity=0.5)
plot += mesh_floor

plot.display()


def update_vis_mesh():
    # Soft body cubes
    simplicits_model.simplicits_scene.sim_z = simplicits_state_0.sim_z
    v0 = simplicits_model.simplicits_scene.get_object_deformed_pts(0, 'rendered')
    v1 = simplicits_model.simplicits_scene.get_object_deformed_pts(1, 'rendered')
    mesh_soft.vertices = np.concatenate(
        [v0.cpu().numpy(), v1.cpu().numpy()]
    ).astype(np.float32)
    # Sand particles (active only)
    points_sand.positions = mpm_state_0.particle_q.numpy()[:mpm_static_start_idx].astype(np.float32)


sim_running = [False]
sim_thread = [None]


def run_sim_loop():
    while sim_running[0]:
        simulate()
        update_vis_mesh()


def on_play(b):
    if not sim_running[0]:
        sim_running[0] = True
        sim_thread[0] = threading.Thread(target=run_sim_loop, daemon=True)
        sim_thread[0].start()


def on_stop(b):
    sim_running[0] = False


def on_reset(b):
    global simplicits_state_0, simplicits_state_1, mpm_state_0, mpm_state_1
    sim_running[0] = False
    if sim_thread[0] is not None:
        sim_thread[0].join()
    simplicits_model.simplicits_scene.reset_scene()
    simplicits_state_0 = simplicits_model.state()
    simplicits_state_1 = simplicits_model.state()
    mpm_state_0 = mpm_model.state()
    mpm_state_1 = mpm_model.state()
    update_vis_mesh()


buttons = [Button(description=x) for x in ["Play", "Stop", "Reset"]]
buttons[0].on_click(on_play)
buttons[1].on_click(on_stop)
buttons[2].on_click(on_reset)

update_vis_mesh()
display(VBox(buttons))